In [1]:
# 한글 폰트 설치 및 설정하기

!pip install statsmodels scikit-learn
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

'sudo'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
'sudo'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
'rm'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [2]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.power import TTestIndPower
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

문제 1 · 펭귄 두 종의 몸무게는 정말 다를까? 난이도 하 · 예상 20분

📖 상황

▸
7장에서 Adelie 수컷만 골라 회귀했을 때 p-값이 0.05를 넘어 기각에 실패했습니다. 그때 우리는 "표본이 작아서"라고 정리했습니다.

▸
이제 반대편에서 물어봅니다. 표본을 아주 크게 만들면 어떤 일이 벌어질까요?

▸
실제 팔머 기지 데이터에서 Adelie와 Chinstrap의 몸무게는 평균 32g밖에 차이나지 않습니다. 펭귄 한 마리 몸무게(약 3,700g)의 1%도 안 되는 차이입니다.

▸
이 32g을 붙잡고 표본만 늘려 가면, p-값은 어디까지 내려갈까요?

🎯 이 문제로 배우는 것

▸
**"표본크기 n이 커지면 p-값은 작아진다"**는 강의 9장의 명제를 실제 데이터로 확인하고, 그래서 **효과크기(Effect Size)**를 함께 봐야 하는 이유를 숫자로 이해합니다.

In [3]:
# 문제 1 · 데이터 준비 — 실행만 하세요
penguins = sns.load_dataset('penguins')
mass = penguins.dropna(subset=['body_mass_g', 'species'])

adelie    = mass.loc[mass['species'] == 'Adelie',    'body_mass_g']
chinstrap = mass.loc[mass['species'] == 'Chinstrap', 'body_mass_g']

print("Adelie    n=%3d   평균 %.1f g   표준편차 %.1f g" % (len(adelie),    adelie.mean(),    adelie.std(ddof=1)))
print("Chinstrap n=%3d   평균 %.1f g   표준편차 %.1f g" % (len(chinstrap), chinstrap.mean(), chinstrap.std(ddof=1)))

diff_g = adelie.mean() - chinstrap.mean()
print("\n두 종의 평균 차이: %.1f g  (Adelie 몸무게의 약 %.1f%%)" % (diff_g, abs(diff_g) / adelie.mean() * 100))

Adelie    n=151   평균 3700.7 g   표준편차 458.6 g
Chinstrap n= 68   평균 3733.1 g   표준편차 384.3 g

두 종의 평균 차이: -32.4 g  (Adelie 몸무게의 약 0.9%)


Q1 · 두 종의 몸무게를 t-검정으로 비교해 봅시다

▸
adelie와 chinstrap의 몸무게에 대해 **이표본 t-검정(Two-sample t-Test)**을 수행하세요. 5-6장 실습에서 쓴 그 검정입니다.

▸
유의수준 0.05를 기준으로 "기각한다 / 기각하지 못한다"까지 문장으로 출력하세요.

In [4]:
from scipy import stats

t_stat, p_value = stats.ttest_ind(adelie, chinstrap)

print("t-통계량 = %.4f" % t_stat)
print("p-값     = %.4f" % p_value)

t-통계량 = -0.5081
p-값     = 0.6119


Q2 · 효과크기(Cohen's d)를 직접 계산해 봅시다

▸ 강의 9장의 공식대로 cohen_d(x, y) 함수를 직접 만드세요. Q3에서 계속 씁니다.

▸ 분자: 두 집단 평균의 차이

▸ 분모: 통합 표준편차 

▸ 만든 함수로 adelie와 chinstrap의 효과크기를 계산해 출력하세요.

▸ Cohen의 관례적 기준(0.2 작음 / 0.5 중간 / 0.8 큼)에서 어디에 놓이는지도 함께 적으세요.

In [5]:
def cohen_d(x, y):
    nx, ny = len(x), len(y)
    sx, sy = np.std(x, ddof=1), np.std(y, ddof=1)
    pooled_s = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    d = (np.mean(x) - np.mean(y)) / pooled_s
    return d

d = cohen_d(adelie, chinstrap)
print("Cohen's d = %.4f" % d)

Cohen's d = -0.0742
